In [10]:
from pprint import pprint
from DbConnector import DbConnector
import statistics


# Part 1: Top 10 Directors by Median Revenue
class MovieQueries:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def query1_top_directors_by_median_revenue(self):
        """
        Find top 10 directors (≥5 movies) by median revenue.
        Report: director name, movie count, median revenue, mean vote_average
        """
        print("\n=== Query 1: Top 10 Directors by Median Revenue ===\n")

        # Step 1: Aggregation pipeline to get all directors with their movies
        pipeline = [
            # Unwind crew array
            {"$unwind": "$crew"},
            # Filter only Directors
            {"$match": {"crew.job": "Director"}},
            # Lookup to join with movies collection
            {
                "$lookup": {
                    "from": "movies",
                    "localField": "_id",
                    "foreignField": "_id",
                    "as": "movie_info",
                }
            },
            # Unwind movie_info (should be 1:1)
            {"$unwind": "$movie_info"},
            # Filter out movies with null/zero revenue
            {"$match": {"movie_info.revenue": {"$ne": None, "$gt": 0}}},
            # Group by director name
            {
                "$group": {
                    "_id": "$crew.name",
                    "movies": {
                        "$push": {
                            "revenue": "$movie_info.revenue",
                            "vote_average": "$movie_info.vote_average",
                        }
                    },
                    "movie_count": {"$sum": 1},
                }
            },
            # Filter directors with ≥ 5 movies
            {"$match": {"movie_count": {"$gte": 5}}},
            # Project needed fields
            {"$project": {"director": "$_id", "movies": 1, "movie_count": 1, "_id": 0}},
        ]

        # Execute aggregation
        results = list(self.db.credits.aggregate(pipeline))

        # Step 2: Calculate median revenue and mean vote_average in Python
        director_stats = []
        for director in results:
            revenues = [m["revenue"] for m in director["movies"]]
            vote_averages = [
                m["vote_average"]
                for m in director["movies"]
                if m["vote_average"] is not None
            ]

            median_revenue = statistics.median(revenues)
            mean_vote_avg = statistics.mean(vote_averages) if vote_averages else None

            director_stats.append(
                {
                    "director": director["director"],
                    "movie_count": director["movie_count"],
                    "median_revenue": median_revenue,
                    "mean_vote_average": (
                        round(mean_vote_avg, 2) if mean_vote_avg else None
                    ),
                }
            )

        # Step 3: Sort by median revenue (descending) and take top 10
        director_stats.sort(key=lambda x: x["median_revenue"], reverse=True)
        top_10 = director_stats[:10]

        # Step 4: Print results
        print(
            f"{'Rank':<5} {'Director':<30} {'Movies':<8} {'Median Revenue':<18} {'Mean Vote Avg':<15}"
        )
        print("-" * 90)
        for i, director in enumerate(top_10, 1):
            print(
                f"{i:<5} {director['director']:<30} {director['movie_count']:<8} "
                f"${director['median_revenue']:>15,.0f}  {director['mean_vote_average']:<15}"
            )

        return top_10

    def close(self):
        self.connection.close_connection()


def main():
    queries = None
    try:
        queries = MovieQueries()

        # Run Query 1
        queries.query1_top_directors_by_median_revenue()

    except Exception as e:
        print(f"ERROR: {e}")
        import traceback

        traceback.print_exc()
    finally:
        if queries:
            queries.close()


if __name__ == "__main__":
    main()


 Connected to MongoDB database: assignment3

=== Query 1: Top 10 Directors by Median Revenue ===

Rank  Director                       Movies   Median Revenue     Mean Vote Avg  
------------------------------------------------------------------------------------------
1     David Yates                    6        $    936,085,968  7.15           
2     Peter Jackson                  11       $    871,368,364  7.27           
3     George Lucas                   6        $    712,398,168  6.88           
4     Brad Bird                      5        $    623,722,818  7.1            
5     Francis Lawrence               6        $    619,388,636  6.8            
6     Tom McGrath                    5        $    532,680,671  6.4            
7     Eric Darnell                   5        $    532,680,671  6.34           
8     Carlos Saldanha                5        $    500,188,435  6.36           
9     Andrew Adamson                 5        $    484,409,218  6.62           
10    John

In [1]:
from pprint import pprint
from DbConnector import DbConnector
from collections import defaultdict


class ActorPairAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def actor_pairs_costarring(self):
        """
        Find actor pairs that co-starred in >= 3 movies together.
        Report: actor names, co-appearance count, average vote_average.
        Sort by co-appearance count (descending).
        """
        print("\n=== Query 2: Actor Pairs Co-starring ===\n")
        
        # Step 1: Fetch all movies with their cast and vote_average
        pipeline = [
            {
                "$lookup": {
                    "from": "movies",
                    "localField": "_id",
                    "foreignField": "_id",
                    "as": "movie_info"
                }
            },
            {"$unwind": "$movie_info"},
            {
                "$project": {
                    "cast": 1,
                    "vote_average": "$movie_info.vote_average",
                    "title": "$movie_info.title"
                }
            }
        ]
        
        movies = list(self.db.credits.aggregate(pipeline))
        
        # Step 2: Build actor pairs with Python
        # pair_data[frozenset({actor1, actor2})] = {movies: [...], vote_averages: [...]}
        pair_data = defaultdict(lambda: {"movies": [], "vote_averages": []})
        
        for movie in movies:
            cast = movie.get("cast", [])
            vote_avg = movie.get("vote_average")
            
            # Get unique actor names (filter out None/empty)
            actors = []
            seen_ids = set()
            for actor in cast:
                actor_id = actor.get("id")
                actor_name = actor.get("name")
                if actor_id and actor_name and actor_id not in seen_ids:
                    actors.append((actor_id, actor_name))
                    seen_ids.add(actor_id)
            
            # Generate all pairs of actors in this movie
            for i in range(len(actors)):
                for j in range(i + 1, len(actors)):
                    actor1_id, actor1_name = actors[i]
                    actor2_id, actor2_name = actors[j]
                    
                    # Use frozenset to ensure (A,B) and (B,A) are treated as same pair
                    pair_key = frozenset({actor1_id})
                    pair_key = frozenset({(actor1_id, actor1_name), (actor2_id, actor2_name)})
                    
                    pair_data[pair_key]["movies"].append(movie["_id"])
                    if vote_avg is not None:
                        pair_data[pair_key]["vote_averages"].append(vote_avg)
        
        # Step 3: Filter pairs with >= 3 co-appearances
        results = []
        for pair_key, data in pair_data.items():
            co_appearance_count = len(data["movies"])
            
            if co_appearance_count >= 3:
                # Extract actor names from frozenset
                actors_list = list(pair_key)
                actor1_name = actors_list[0][1]
                actor2_name = actors_list[1][1]
                
                # Calculate average vote_average
                avg_vote = (sum(data["vote_averages"]) / len(data["vote_averages"]) 
                           if data["vote_averages"] else None)
                
                results.append({
                    "actor1": actor1_name,
                    "actor2": actor2_name,
                    "co_appearances": co_appearance_count,
                    "avg_vote_average": round(avg_vote, 2) if avg_vote else None
                })
        
        # Step 4: Sort by co-appearance count (descending)
        results.sort(key=lambda x: x["co_appearances"], reverse=True)
        
        # Step 5: Print results
        print(f"{'Rank':<5} {'Actor 1':<30} {'Actor 2':<30} {'Co-appearances':<15} {'Avg Vote':<10}")
        print("-" * 95)
        
        for i, pair in enumerate(results[:20], 1):  # Show top 20
            print(f"{i:<5} {pair['actor1']:<30} {pair['actor2']:<30} "
                  f"{pair['co_appearances']:<15} {pair['avg_vote_average']:<10}")
        
        print(f"\nTotal pairs found: {len(results)}")
        
        return results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = ActorPairAnalysis()
        results = program.actor_pairs_costarring()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 2: Actor Pairs Co-starring ===

Rank  Actor 1                        Actor 2                        Co-appearances  Avg Vote  
-----------------------------------------------------------------------------------------------
1     Leo Gorcey                     Huntz Hall                     35              6.35      
2     Charlie Chaplin                Edna Purviance                 33              6.46      
3     Stan Laurel                    Oliver Hardy                   31              6.35      
4     Rob Paulsen                    Jeff Bennett                   27              6.23      
5     Bud Abbott                     Lou Costello                   27              6.47      
6     Grey Griffin                   Frank Welker                   25              6.71      
7     Barbara Hale                   Raymond Burr                   25              5.93      
8     John Wayne                     Paul Fix            

In [2]:
from pprint import pprint
from DbConnector import DbConnector


class GenreBreadthAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def top_actors_by_genre_breadth(self):
        """
        Find top 10 actors (with >= 10 credited movies) with widest genre breadth.
        Report: actor name, number of distinct genres, up to 5 example genres.
        """
        print("\n=== Query 3: Actors with Widest Genre Breadth ===\n")
        
        # MongoDB aggregation pipeline
        pipeline = [
            # Step 1: Unwind cast array
            {"$unwind": "$cast"},
            
            # Step 2: Lookup movies to get genres
            {
                "$lookup": {
                    "from": "movies",
                    "localField": "_id",
                    "foreignField": "_id",
                    "as": "movie_info"
                }
            },
            {"$unwind": "$movie_info"},
            
            # Step 3: Unwind genres array
            {"$unwind": "$movie_info.genres"},
            
            # Step 4: Group by actor
            {
                "$group": {
                    "_id": {
                        "actor_id": "$cast.id",
                        "actor_name": "$cast.name"
                    },
                    "genres": {"$addToSet": "$movie_info.genres.name"},
                    "movie_count": {"$addToSet": "$_id"}  # Count distinct movies
                }
            },
            
            # Step 5: Project and calculate counts
            {
                "$project": {
                    "actor_id": "$_id.actor_id",
                    "actor_name": "$_id.actor_name",
                    "genres": 1,
                    "genre_count": {"$size": "$genres"},
                    "movie_count": {"$size": "$movie_count"},
                    "_id": 0
                }
            },
            
            # Step 6: Filter actors with >= 10 movies
            {"$match": {"movie_count": {"$gte": 10}}},
            
            # Step 7: Sort by genre count descending
            {"$sort": {"genre_count": -1}},
            
            # Step 8: Limit to top 10
            {"$limit": 10}
        ]
        
        results = list(self.db.credits.aggregate(pipeline))
        
        # Print results
        print(f"{'Rank':<5} {'Actor':<35} {'Movies':<8} {'Genres':<8} {'Example Genres':<50}")
        print("-" * 110)
        
        for i, actor in enumerate(results, 1):
            # Get up to 5 example genres
            example_genres = ', '.join(actor['genres'][:5])
            
            print(f"{i:<5} {actor['actor_name']:<35} {actor['movie_count']:<8} "
                  f"{actor['genre_count']:<8} {example_genres:<50}")
        
        return results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = GenreBreadthAnalysis()
        results = program.top_actors_by_genre_breadth()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 3: Actors with Widest Genre Breadth ===

Rank  Actor                               Movies   Genres   Example Genres                                    
--------------------------------------------------------------------------------------------------------------
1     Martin Sheen                        76       20       Fantasy, Music, Comedy, TV Movie, Adventure       
2     Christopher Lee                     145      20       Thriller, Family, Music, Drama, Animation         
3     Charlton Heston                     68       20       Thriller, Western, Animation, Music, War          
4     Ned Beatty                          70       20       Comedy, TV Movie, Crime, Romance, Horror          
5     Stacy Keach                         59       20       Fantasy, History, Science Fiction, Horror, Crime  
6     Eddie Albert                        47       20       Music, Action, War, Foreign, Animation            
7     Michael Ga

In [3]:
from pprint import pprint
from DbConnector import DbConnector
import statistics


class CollectionAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def top_collections_by_revenue(self):
        """
        Find top 10 film collections (>= 3 movies) by total revenue.
        Report: collection name, movie count, total revenue, median vote_average,
        earliest -> latest release date.
        """
        print("\n=== Query 4: Top Collections by Total Revenue ===\n")
        
        # MongoDB aggregation pipeline
        pipeline = [
            # Step 1: Filter movies that belong to a collection
            {
                "$match": {
                    "belongs_to_collection": {"$ne": None},
                    "belongs_to_collection.name": {"$exists": True, "$ne": None}
                }
            },
            
            # Step 2: Group by collection name
            {
                "$group": {
                    "_id": "$belongs_to_collection.name",
                    "movie_count": {"$sum": 1},
                    "total_revenue": {"$sum": "$revenue"},
                    "vote_averages": {"$push": "$vote_average"},
                    "release_dates": {"$push": "$release_date"},
                    "revenues": {"$push": "$revenue"}  # For filtering later
                }
            },
            
            # Step 3: Filter collections with >= 3 movies
            {"$match": {"movie_count": {"$gte": 3}}},
            
            # Step 4: Sort by total revenue descending
            {"$sort": {"total_revenue": -1}},
            
            # Step 5: Limit to top 10
            {"$limit": 10}
        ]
        
        results = list(self.db.movies.aggregate(pipeline))
        
        # Process results in Python for median and date range
        processed_results = []
        for collection in results:
            # Calculate median vote_average (filter out None values)
            vote_avgs = [v for v in collection['vote_averages'] if v is not None]
            median_vote = statistics.median(vote_avgs) if vote_avgs else None
            
            # Get earliest and latest release dates (filter out None)
            dates = [d for d in collection['release_dates'] if d is not None]
            dates.sort()
            earliest_date = dates[0] if dates else None
            latest_date = dates[-1] if dates else None
            
            processed_results.append({
                "collection_name": collection['_id'],
                "movie_count": collection['movie_count'],
                "total_revenue": collection['total_revenue'],
                "median_vote_average": round(median_vote, 2) if median_vote else None,
                "earliest_release": earliest_date,
                "latest_release": latest_date
            })
        
        # Print results
        print(f"{'Rank':<5} {'Collection':<40} {'Movies':<8} {'Total Revenue':<18} "
              f"{'Median Vote':<12} {'Release Span':<25}")
        print("-" * 115)
        
        for i, col in enumerate(processed_results, 1):
            date_span = f"{col['earliest_release']} → {col['latest_release']}" if col['earliest_release'] else "N/A"
            
            print(f"{i:<5} {col['collection_name']:<40} {col['movie_count']:<8} "
                  f"${col['total_revenue']:>15,.0f}  {col['median_vote_average']:<12} {date_span:<25}")
        
        return processed_results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = CollectionAnalysis()
        results = program.top_collections_by_revenue()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 4: Top Collections by Total Revenue ===

Rank  Collection                               Movies   Total Revenue      Median Vote  Release Span             
-------------------------------------------------------------------------------------------------------------------
1     Harry Potter Collection                  8        $  7,707,367,425  7.5          2001-11-16 → 2011-07-07  
2     Star Wars Collection                     8        $  7,434,494,790  7.45         1977-05-25 → 2016-12-14  
3     James Bond Collection                    26       $  7,106,970,239  6.3          1962-10-04 → 2015-10-26  
4     The Fast and the Furious Collection      8        $  5,125,098,793  6.65         2001-06-22 → 2017-04-12  
5     Pirates of the Caribbean Collection      5        $  4,521,576,826  6.9          2003-07-09 → 2017-05-23  
6     Transformers Collection                  5        $  4,366,101,244  6.1          2007-06-27 → 2017-06-2

In [4]:
from pprint import pprint
from DbConnector import DbConnector
import statistics


class DecadeGenreAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def median_runtime_by_decade_genre(self):
        """
        Find median runtime and movie count by decade and primary genre.
        Sort by decade, then median runtime (descending).
        """
        print("\n=== Query 5: Median Runtime by Decade and Primary Genre ===\n")
        
        # MongoDB aggregation pipeline
        pipeline = [
            # Step 1: Filter movies with valid release_date, runtime, and genres
            {
                "$match": {
                    "release_date": {"$ne": None},
                    "runtime": {"$ne": None, "$gt": 0},
                    "genres": {"$exists": True, "$ne": []}
                }
            },
            
            # Step 2: Extract decade and primary genre
            {
                "$project": {
                    "decade": {
                        "$multiply": [
                            {"$floor": {"$divide": [{"$year": {"$dateFromString": {"dateString": "$release_date"}}}, 10]}},
                            10
                        ]
                    },
                    "primary_genre": {"$arrayElemAt": ["$genres.name", 0]},
                    "runtime": 1
                }
            },
            
            # Step 3: Group by decade and primary genre
            {
                "$group": {
                    "_id": {
                        "decade": "$decade",
                        "primary_genre": "$primary_genre"
                    },
                    "runtimes": {"$push": "$runtime"},
                    "movie_count": {"$sum": 1}
                }
            },
            
            # Step 4: Sort by decade, then we'll sort by median runtime in Python
            {"$sort": {"_id.decade": 1}}
        ]
        
        results = list(self.db.movies.aggregate(pipeline))
        
        # Calculate median runtime in Python
        processed_results = []
        for group in results:
            runtimes = group['runtimes']
            median_runtime = statistics.median(runtimes) if runtimes else None
            
            processed_results.append({
                "decade": f"{group['_id']['decade']}s",
                "primary_genre": group['_id']['primary_genre'],
                "movie_count": group['movie_count'],
                "median_runtime": round(median_runtime, 1) if median_runtime else None
            })
        
        # Sort by decade, then median runtime (descending)
        processed_results.sort(key=lambda x: (x['decade'], -x['median_runtime'] if x['median_runtime'] else 0))
        
        # Print results
        print(f"{'Decade':<10} {'Primary Genre':<25} {'Movie Count':<12} {'Median Runtime (min)':<20}")
        print("-" * 70)
        
        for result in processed_results:
            print(f"{result['decade']:<10} {result['primary_genre']:<25} "
                  f"{result['movie_count']:<12} {result['median_runtime']:<20}")
        
        return processed_results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = DecadeGenreAnalysis()
        results = program.median_runtime_by_decade_genre()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 5: Median Runtime by Decade and Primary Genre ===

Decade     Primary Genre             Movie Count  Median Runtime (min)
----------------------------------------------------------------------
1870.0s    Documentary               2            1.0                 
1880.0s    Documentary               4            1.0                 
1890.0s    Fantasy                   8            1.5                 
1890.0s    Action                    2            1.0                 
1890.0s    Family                    1            1.0                 
1890.0s    Comedy                    9            1.0                 
1890.0s    Drama                     2            1.0                 
1890.0s    History                   1            1.0                 
1890.0s    Horror                    3            1.0                 
1890.0s    Documentary               27           1.0                 
1900.0s    History                   1    

In [5]:
from pprint import pprint
from DbConnector import DbConnector


class GenderAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def female_proportion_by_decade(self):
        """
        Calculate proportion of female cast in top-billed 5 actors per movie.
        Aggregate by decade, sort by average female proportion (descending).
        Gender: 1 = female, 2 = male, 0/null = unknown (ignored).
        """
        print("\n=== Query 6: Female Cast Proportion by Decade ===\n")
        
        # MongoDB aggregation pipeline
        pipeline = [
            # Step 1: Lookup movies to get release_date
            {
                "$lookup": {
                    "from": "movies",
                    "localField": "_id",
                    "foreignField": "_id",
                    "as": "movie_info"
                }
            },
            {"$unwind": "$movie_info"},
            
            # Step 2: Filter movies with valid release_date
            {
                "$match": {
                    "movie_info.release_date": {"$ne": None}
                }
            },
            
            # Step 3: Project top 5 cast and decade
            {
                "$project": {
                    "decade": {
                        "$multiply": [
                            {"$floor": {
                                "$divide": [
                                    {"$year": {"$dateFromString": {"dateString": "$movie_info.release_date"}}},
                                    10
                                ]
                            }},
                            10
                        ]
                    },
                    "top5_cast": {
                        "$slice": [
                            {
                                "$filter": {
                                    "input": {
                                        "$sortArray": {
                                            "input": "$cast",
                                            "sortBy": {"order": 1}
                                        }
                                    },
                                    "as": "actor",
                                    "cond": {"$in": ["$$actor.gender", [1, 2]]}  # Only known genders
                                }
                            },
                            5
                        ]
                    }
                }
            },
            
            # Step 4: Calculate female proportion per movie
            {
                "$project": {
                    "decade": 1,
                    "cast_count": {"$size": "$top5_cast"},
                    "female_count": {
                        "$size": {
                            "$filter": {
                                "input": "$top5_cast",
                                "as": "actor",
                                "cond": {"$eq": ["$$actor.gender", 1]}
                            }
                        }
                    }
                }
            },
            
            # Step 5: Filter movies with at least 1 cast member
            {"$match": {"cast_count": {"$gt": 0}}},
            
            # Step 6: Calculate proportion per movie
            {
                "$project": {
                    "decade": 1,
                    "female_proportion": {
                        "$divide": ["$female_count", "$cast_count"]
                    }
                }
            },
            
            # Step 7: Group by decade
            {
                "$group": {
                    "_id": "$decade",
                    "avg_female_proportion": {"$avg": "$female_proportion"},
                    "movie_count": {"$sum": 1}
                }
            },
            
            # Step 8: Sort by average female proportion (descending)
            {"$sort": {"avg_female_proportion": -1}},
            
            # Step 9: Format output
            {
                "$project": {
                    "decade": {"$concat": [{"$toString": "$_id"}, "s"]},
                    "avg_female_proportion": {"$multiply": ["$avg_female_proportion", 100]},  # Convert to percentage
                    "movie_count": 1,
                    "_id": 0
                }
            }
        ]
        
        results = list(self.db.credits.aggregate(pipeline))
        
        # Print results
        print(f"{'Decade':<10} {'Avg Female %':<15} {'Movie Count':<12}")
        print("-" * 40)
        
        for result in results:
            print(f"{result['decade']:<10} {result['avg_female_proportion']:>13.2f}%  {result['movie_count']:<12}")
        
        return results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = GenderAnalysis()
        results = program.female_proportion_by_decade()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 6: Female Cast Proportion by Decade ===

Decade     Avg Female %    Movie Count 
----------------------------------------
1870s             100.00%  1           
2020s              80.00%  1           
1890s              50.00%  2           
2010s              37.14%  10760       
2000s              35.85%  9800        
1930s              34.95%  1264        
1900s              34.49%  13          
1940s              34.39%  1438        
1990s              33.17%  5227        
1950s              32.12%  2007        
1910s              31.81%  150         
1960s              31.61%  2442        
1980s              31.41%  3632        
1920s              30.76%  378         
1970s              30.30%  3203        
 Connection to MongoDB closed


In [6]:
from pprint import pprint
from DbConnector import DbConnector


class NoirSearchAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def top_noir_movies(self):
        """
        Find top 20 movies matching "noir" or "neo-noir" in overview/tagline.
        Filter: vote_count >= 50
        Sort by: vote_average (descending)
        Return: title, year, vote_average, vote_count
        """
        print("\n=== Query 7: Top Noir Movies ===\n")
        
        # Step 1: Create text index (if not exists)
        try:
            self.db.movies.create_index([
                ("overview", "text"),
                ("tagline", "text")
            ], name="overview_tagline_text")
            print("Text index created/verified")
        except Exception as e:
            print(f"Index already exists or error: {e}")
        
        # Step 2: MongoDB aggregation pipeline with text search
        pipeline = [
            # Text search for "noir" or "neo-noir"
            {
                "$match": {
                    "$text": {"$search": "noir neo-noir"},
                    "vote_count": {"$gte": 50}
                }
            },
            
            # Extract year from release_date
            {
                "$project": {
                    "title": 1,
                    "year": {
                        "$cond": {
                            "if": {"$ne": ["$release_date", None]},
                            "then": {"$year": {"$dateFromString": {"dateString": "$release_date"}}},
                            "else": None
                        }
                    },
                    "vote_average": 1,
                    "vote_count": 1,
                    "textScore": {"$meta": "textScore"}  # Get relevance score
                }
            },
            
            # Sort by vote_average (descending), then textScore for ties
            {"$sort": {"vote_average": -1, "textScore": -1}},
            
            # Limit to top 20
            {"$limit": 20},
            
            # Clean up output
            {
                "$project": {
                    "title": 1,
                    "year": 1,
                    "vote_average": 1,
                    "vote_count": 1,
                    "_id": 0
                }
            }
        ]
        
        results = list(self.db.movies.aggregate(pipeline))
        
        # Print results
        print(f"{'Rank':<5} {'Title':<50} {'Year':<6} {'Vote Avg':<10} {'Vote Count':<10}")
        print("-" * 85)
        
        for i, movie in enumerate(results, 1):
            year = movie.get('year') or 'N/A'
            print(f"{i:<5} {movie['title']:<50} {year:<6} "
                  f"{movie['vote_average']:<10.1f} {movie['vote_count']:<10}")
        
        return results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = NoirSearchAnalysis()
        results = program.top_noir_movies()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 7: Top Noir Movies ===

Text index created/verified
Rank  Title                                              Year   Vote Avg   Vote Count
-------------------------------------------------------------------------------------
1     Rome, Open City                                    1945   7.9        123       
2     Akira                                              1988   7.8        792       
3     The Bad Sleep Well                                 1960   7.7        57        
4     Drunken Angel                                      1948   7.7        54        
5     Elevator to the Gallows                            1958   7.6        85        
6     Adam's Apples                                      2005   7.4        190       
7     The Matrix Reloaded                                2003   6.7        3500      
8     The Matrix Revolutions                             2003   6.4        3155      
9     Frontier(s)                

In [7]:
from pprint import pprint
from DbConnector import DbConnector
from collections import defaultdict


class DirectorActorPairAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def top_director_actor_pairs(self):
        """
        Find top 20 director-actor pairs with >= 3 collaborations and highest mean vote_average.
        Filter: only movies with vote_count >= 100
        Report: director, actor, collaboration count, mean vote_average, mean revenue
        """
        print("\n=== Query 8: Top Director-Actor Pairs ===\n")
        
        # Step 1: Fetch credits with movie info
        pipeline = [
            # Lookup movies to get vote_count, vote_average, revenue
            {
                "$lookup": {
                    "from": "movies",
                    "localField": "_id",
                    "foreignField": "_id",
                    "as": "movie_info"
                }
            },
            {"$unwind": "$movie_info"},
            
            # Filter movies with vote_count >= 100
            {"$match": {"movie_info.vote_count": {"$gte": 100}}},
            
            # Project needed fields
            {
                "$project": {
                    "cast": 1,
                    "crew": 1,
                    "vote_average": "$movie_info.vote_average",
                    "revenue": "$movie_info.revenue",
                    "vote_count": "$movie_info.vote_count"
                }
            }
        ]
        
        movies = list(self.db.credits.aggregate(pipeline))
        
        # Step 2: Build director-actor pairs in Python
        # pair_data[(director_name, actor_name)] = {movies: [...], vote_averages: [...], revenues: [...]}
        pair_data = defaultdict(lambda: {"movies": [], "vote_averages": [], "revenues": []})
        
        for movie in movies:
            # Extract directors
            directors = [
                crew.get("name") 
                for crew in movie.get("crew", []) 
                if crew.get("job") == "Director" and crew.get("name")
            ]
            
            # Extract actors (with valid names)
            actors = [
                actor.get("name")
                for actor in movie.get("cast", [])
                if actor.get("name")
            ]
            
            vote_avg = movie.get("vote_average")
            revenue = movie.get("revenue")
            movie_id = movie.get("_id")
            
            # Create all director-actor pairs for this movie
            for director in directors:
                for actor in actors:
                    pair_key = (director, actor)
                    pair_data[pair_key]["movies"].append(movie_id)
                    
                    if vote_avg is not None:
                        pair_data[pair_key]["vote_averages"].append(vote_avg)
                    
                    if revenue is not None and revenue > 0:
                        pair_data[pair_key]["revenues"].append(revenue)
        
        # Step 3: Filter pairs with >= 3 collaborations and calculate statistics
        results = []
        for pair_key, data in pair_data.items():
            collaboration_count = len(data["movies"])
            
            if collaboration_count >= 3:
                director, actor = pair_key
                
                # Calculate mean vote_average
                mean_vote_avg = (
                    sum(data["vote_averages"]) / len(data["vote_averages"])
                    if data["vote_averages"] else None
                )
                
                # Calculate mean revenue
                mean_revenue = (
                    sum(data["revenues"]) / len(data["revenues"])
                    if data["revenues"] else None
                )
                
                results.append({
                    "director": director,
                    "actor": actor,
                    "collaboration_count": collaboration_count,
                    "mean_vote_average": round(mean_vote_avg, 2) if mean_vote_avg else None,
                    "mean_revenue": round(mean_revenue, 0) if mean_revenue else None
                })
        
        # Step 4: Sort by mean vote_average (descending) and take top 20
        results.sort(key=lambda x: x["mean_vote_average"] if x["mean_vote_average"] else 0, reverse=True)
        top_20 = results[:20]
        
        # Print results
        print(f"{'Rank':<5} {'Director':<30} {'Actor':<30} {'Films':<7} {'Mean Vote':<11} {'Mean Revenue':<15}")
        print("-" * 105)
        
        for i, pair in enumerate(top_20, 1):
            revenue_str = f"${pair['mean_revenue']:,.0f}" if pair['mean_revenue'] else "N/A"
            print(f"{i:<5} {pair['director']:<30} {pair['actor']:<30} "
                  f"{pair['collaboration_count']:<7} {pair['mean_vote_average']:<11} {revenue_str:<15}")
        
        return top_20

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = DirectorActorPairAnalysis()
        results = program.top_director_actor_pairs()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 8: Top Director-Actor Pairs ===

Rank  Director                       Actor                          Films   Mean Vote   Mean Revenue   
---------------------------------------------------------------------------------------------------------
1     Charlie Chaplin                Hank Mann                      3       8.13        $6,506,394     
2     Akira Kurosawa                 Eijirô Tôno                    3       8.13        $271,841       
3     Akira Kurosawa                 Minoru Itô                     3       8.13        $271,841       
4     Akira Kurosawa                 Haruo Suzuki                   3       8.13        $271,841       
5     Quentin Tarantino              Harvey Keitel                  3       8.1         $182,573,606   
6     Francis Ford Coppola           John Cazale                    3       8.1         $99,009,751    
7     Akira Kurosawa                 Atsushi Watanabe               3       8.

In [8]:
from pprint import pprint
from DbConnector import DbConnector


class LanguageAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def top_non_english_us_languages(self):
        """
        Find top 10 original languages (not English) for movies with US production.
        Filter: original_language != "en" AND (US production company OR US country)
        Report: language, count, one example title
        """
        print("\n=== Query 9: Top Non-English Languages in US Productions ===\n")
        
        # MongoDB aggregation pipeline
        pipeline = [
            # Step 1: Filter non-English movies with US involvement
            {
                "$match": {
                    "original_language": {"$ne": "en"},
                    "$or": [
                        {"production_companies.name": {"$regex": "United States", "$options": "i"}},
                        {"production_countries.name": {"$regex": "United States", "$options": "i"}},
                        {"production_countries.iso_3166_1": "US"}
                    ]
                }
            },
            
            # Step 2: Group by original language
            {
                "$group": {
                    "_id": "$original_language",
                    "count": {"$sum": 1},
                    "example_title": {"$first": "$title"}  # Get one example
                }
            },
            
            # Step 3: Sort by count (descending)
            {"$sort": {"count": -1}},
            
            # Step 4: Limit to top 10
            {"$limit": 10},
            
            # Step 5: Format output
            {
                "$project": {
                    "language": "$_id",
                    "count": 1,
                    "example_title": 1,
                    "_id": 0
                }
            }
        ]
        
        results = list(self.db.movies.aggregate(pipeline))
        
        # Print results
        print(f"{'Rank':<5} {'Language':<12} {'Count':<8} {'Example Title':<60}")
        print("-" * 90)
        
        for i, lang in enumerate(results, 1):
            print(f"{i:<5} {lang['language']:<12} {lang['count']:<8} {lang['example_title']:<60}")
        
        return results

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = LanguageAnalysis()
        results = program.top_non_english_us_languages()
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 9: Top Non-English Languages in US Productions ===

Rank  Language     Count    Example Title                                               
------------------------------------------------------------------------------------------
1     fr           112      Wings of Courage                                            
2     es           72       Bitter Sugar                                                
3     it           56       Frankie Starlight                                           
4     de           51       Cold Fever                                                  
5     ja           30       Godzilla 1985                                               
6     pt           15       Senseless                                                   
7     xx           14       Quest for Fire                                              
8     nl           12       Come On, Rangers                                            
9

In [11]:
from pprint import pprint
from DbConnector import DbConnector
import statistics


class UserAnalysis:
    def __init__(self):
        self.connection = DbConnector()
        self.db = self.connection.db

    def analyze_users(self):
        """
        For each user (with >= 20 ratings), calculate:
        - Ratings count
        - Population variance of ratings
        - Number of distinct genres rated
        
        List top 10 by genre diversity and top 10 by variance separately.
        """
        print("\n=== Query 10: User Rating Statistics ===\n")
        
        # Step 1: Aggregate user ratings with movie genres
        pipeline = [
            # Lookup movies to get genres
            {
                "$lookup": {
                    "from": "movies",
                    "localField": "tmdbId",
                    "foreignField": "_id",
                    "as": "movie_info"
                }
            },
            {"$unwind": "$movie_info"},
            
            # Unwind genres
            {"$unwind": {"path": "$movie_info.genres", "preserveNullAndEmptyArrays": False}},
            
            # Group by user
            {
                "$group": {
                    "_id": "$userId",
                    "ratings": {"$push": "$rating"},
                    "genres": {"$addToSet": "$movie_info.genres.name"}
                }
            },
            
            # Calculate counts
            {
                "$project": {
                    "userId": "$_id",
                    "ratings": 1,
                    "rating_count": {"$size": "$ratings"},
                    "genre_count": {"$size": "$genres"},
                    "_id": 0
                }
            },
            
            # Filter users with >= 20 ratings
            {"$match": {"rating_count": {"$gte": 20}}}
        ]
        
        users = list(self.db.ratings.aggregate(pipeline))
        
        print(f"Analyzing {len(users)} users with >= 20 ratings...")
        
        # Step 2: Calculate variance in Python
        user_stats = []
        for user in users:
            ratings = user["ratings"]
            
            # Calculate population variance
            variance = statistics.pvariance(ratings) if len(ratings) > 1 else 0
            
            user_stats.append({
                "userId": user["userId"],
                "rating_count": user["rating_count"],
                "genre_count": user["genre_count"],
                "rating_variance": round(variance, 4)
            })
        
        # Step 3: Top 10 by genre diversity
        top_genre_diverse = sorted(user_stats, key=lambda x: x["genre_count"], reverse=True)[:10]
        
        print("\n--- Top 10 Most Genre-Diverse Users ---\n")
        print(f"{'Rank':<5} {'User ID':<10} {'Rating Count':<15} {'Genre Count':<12} {'Rating Variance':<15}")
        print("-" * 70)
        
        for i, user in enumerate(top_genre_diverse, 1):
            print(f"{i:<5} {user['userId']:<10} {user['rating_count']:<15} "
                  f"{user['genre_count']:<12} {user['rating_variance']:<15}")
        
        # Step 4: Top 10 by rating variance
        top_variance = sorted(user_stats, key=lambda x: x["rating_variance"], reverse=True)[:10]
        
        print("\n--- Top 10 Highest-Variance Users ---\n")
        print(f"{'Rank':<5} {'User ID':<10} {'Rating Count':<15} {'Genre Count':<12} {'Rating Variance':<15}")
        print("-" * 70)
        
        for i, user in enumerate(top_variance, 1):
            print(f"{i:<5} {user['userId']:<10} {user['rating_count']:<15} "
                  f"{user['genre_count']:<12} {user['rating_variance']:<15}")
        
        return {
            "top_genre_diverse": top_genre_diverse,
            "top_variance": top_variance,
            "total_users": len(user_stats)
        }

    def close(self):
        self.connection.close_connection()


def main():
    program = None
    try:
        program = UserAnalysis()
        results = program.analyze_users()
        
        print(f"\n--- Summary ---")
        print(f"Total users analyzed: {results['total_users']}")
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if program:
            program.close()


if __name__ == '__main__':
    main()

 Connected to MongoDB database: assignment3

=== Query 10: User Rating Statistics ===

Analyzing 240524 users with >= 20 ratings...

--- Top 10 Most Genre-Diverse Users ---

Rank  User ID    Rating Count    Genre Count  Rating Variance
----------------------------------------------------------------------
1     229        3382            20           1.1636         
2     231        3681            20           0.39           
3     567        2456            20           1.2925         
4     741        7487            20           0.4703         
5     836        4241            20           0.5048         
6     924        2582            20           1.0289         
7     1104       3096            20           0.7188         
8     1145       1517            20           1.3906         
9     1214       477             20           0.8207         
10    1380       2945            20           0.3707         

--- Top 10 Highest-Variance Users ---

Rank  User ID    Rating Count    